# FindWord
## Find a typed Urdu phrase in a recording

Type **moong phali** or **مونگ پھلی** and rank recordings by where the phrase sounds most likely. This notebook contains a small trainable acoustic model, a character-level phrase matcher, and a pilot derived from eight real FLEURS Urdu development recordings. The researcher supplies or trains the weights. Use Python 3.11 or newer, PyTorch 2.1 or newer, NumPy, SciPy and IPython. The default recipe is designed for a CPU or GPU notebook.

## 1 · Build dataset

Each of the eight recordings comes from a different FLEURS sentence ID. A positive label means the complete sequence of words appears in its published transcript. Four extra queries have no match among these eight. Three of them share a word with a positive query. The two Roman spellings are illustrative aliases. Each row corresponds to one canonical query and one recording, irrespective of the number of spellings.


In [ ]:
from pathlib import Path
import csv
import hashlib
import itertools
import math
import os
import random
import re
import shutil
import tarfile
import tempfile
import unicodedata
import urllib.request
import warnings
from collections import Counter, defaultdict

import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly

WORK = Path("findword_work")
WORK.mkdir(exist_ok=True)
SEED = 27

# Each tuple contains the sentence ID, WAV filename, native query and two pilot aliases.
EXAMPLES = [
    ("1581", "833305250583810251.wav", "سطح زمین", "satah zameen", "sath zameen"),
    ("1531", "7403792130098535949.wav", "بحر ہند", "bahr e hind", "behr-e-hind"),
    ("1587", "7578294615762591907.wav", "جولائی", "julai", "julaai"),
    ("1643", "1549359331696083700.wav", "جنگلی حیات", "jangli hayat", "jangli hayaat"),
    ("1604", "13224202044146426739.wav", "قومی سلامتی", "qaumi salamti", "qoumi salaamti"),
    ("1620", "6976654990101560269.wav", "مونگ پھلی", "moong phali", "mung phali"),
    ("1518", "6356088897638302660.wav", "ہانگ کانگ", "hong kong", "haang kaang"),
    ("1547", "1110063729369159657.wav", "پاور پلانٹ", "power plant", "pawar plant"),
]
ABSENT = [
    ("قومی اسمبلی", "qaumi assembly", "qoumi assembli"),
    ("پاور ہاؤس", "power house", "pawar haus"),
    ("بجلی کا بل", "bijli ka bill", "bijlee ka bil"),
    ("مونگ دال", "moong daal", "mung dal"),
]
QUERIES = [x[2:] for x in EXAMPLES] + ABSENT
SEED_FIELDS = ("query_id", "urdu", "roman_1", "roman_2", "dataset", "split", "sentence_id",
               "audio_file", "present_in_reference", "start_s", "end_s")
PILOT = [
    dict(zip(SEED_FIELDS, (f"q{j:02}", ur, a, b, "google/fleurs:ur_pk", "validation",
                           sid, wav, int(ur == expected), "", "")))
    for j, (ur, a, b) in enumerate(QUERIES, 1)
    for sid, wav, expected, *_ in EXAMPLES
]

def write_rows(path, fields, rows):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

write_rows(WORK / "findword_seed.csv", SEED_FIELDS, PILOT)
print(f"{len(PILOT)} canonical pairs; {sum(r['present_in_reference'] for r in PILOT)} positive.")


## 2 · Fetch and check the source recordings

The loader reads Google's original seven-column TSV. It verifies SHA-256 for each audio archive and copies only the required WAV members. The seed CSV omits word boundaries so that a listener can mark the true start and end in seconds. Play the example and check the phrase by ear.


In [ ]:
BASE = "https://huggingface.co/datasets/google/fleurs/resolve/main/data/ur_pk"
ARCHIVE_SHA = {
    "dev": "ee0a7a18242a70ea9088e9618337e4835ce1972557bdc2cd3d500555ce89520f",
    "train": "00fbfdd4b98fa4c264b843355aa7450a3922642b73cb13e54ffa8950c4cdd93a",
}
FOLDS = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ـ": "", "\u200c": " ", "\u200d": ""})

def normalize(value):
    value = unicodedata.normalize("NFKC", value).casefold().translate(FOLDS)
    value = "".join(c for c in value if not unicodedata.category(c).startswith("M"))
    return " ".join("".join(c if c.isalnum() else " " for c in value).split())

def phrase_in(query, transcript):
    q = normalize(query)
    return bool(q) and (" " + q + " ") in (" " + normalize(transcript) + " ")

def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def download(url, dest, expected=None):
    if dest.exists() and (expected is None or digest(dest) == expected):
        return dest
    temp = None
    try:
        with tempfile.NamedTemporaryFile(dir=dest.parent, delete=False) as out:
            temp = Path(out.name)
            request = urllib.request.Request(url, headers={"User-Agent": "findword-research/0.1"})
            with urllib.request.urlopen(request, timeout=90) as response:
                shutil.copyfileobj(response, out, length=1024 * 1024)
        if expected and digest(temp) != expected:
            raise ValueError(f"Checksum mismatch for {dest.name}. Check the source and retry.")
        os.replace(temp, dest)
    finally:
        if temp and temp.exists():
            temp.unlink()
    return dest

def read_manifest(split):
    path = download(f"{BASE}/{split}.tsv", WORK / f"{split}.tsv")
    rows = {}
    for line in path.read_text(encoding="utf-8-sig").split("\n"):
        if not line.strip():
            continue
        # Some raw transcripts contain legitimate quote characters.
        sid, wav, raw, normalized, chars, samples, gender = line.split("\t")
        if wav in rows:
            raise ValueError(f"Duplicate audio filename {wav}")
        rows[wav] = {"sentence_id": sid, "text": raw, "samples": int(samples)}
    return rows

def fetch_wavs(split, names):
    target = WORK / split
    target.mkdir(exist_ok=True)
    missing = {name for name in names if not (target / name).is_file()}
    if missing:
        archive = download(f"{BASE}/audio/{split}.tar.gz", WORK / f"{split}.tar.gz", ARCHIVE_SHA[split])
        with tarfile.open(archive, "r|gz") as source:
            for member in source:
                name = Path(member.name).name
                if not member.isfile() or name not in missing:
                    continue
                with source.extractfile(member) as src, tempfile.NamedTemporaryFile(dir=target, delete=False) as dst:
                    tmp = Path(dst.name)
                    try:
                        shutil.copyfileobj(src, dst)
                    except BaseException:
                        tmp.unlink(missing_ok=True)
                        raise
                os.replace(tmp, target / name)
                missing.remove(name)
                if not missing:
                    break
    if missing:
        raise FileNotFoundError(f"Missing expected archive members: {sorted(missing)}")
    return target

dev = read_manifest("dev")
for row in PILOT:
    ref = dev[row["audio_file"]]
    assert ref["sentence_id"] == row["sentence_id"]
    assert phrase_in(row["urdu"], ref["text"]) == bool(row["present_in_reference"])
print(f"Verified every pilot pair against the source. Development split has {len(dev)} recordings.")
dev_dir = fetch_wavs("dev", [x[1] for x in EXAMPLES])


In [ ]:
from IPython.display import Audio, display

demo_wav = "6976654990101560269.wav"
print("Listen for مونگ پھلی (moong phali).")
display(Audio(filename=str(dev_dir / demo_wav)))


## 3 · Compare against transcript search

For the first baseline, transcribe these eight WAV files with your existing Urdu Zipformer. Put the predictions in `findword_work/zipformer_predictions.csv` with the headers `audio_file,text`. The first field is the original WAV basename. The second is your model's transcript. The code checks for all eight files.

Both the acoustic and the transcript baseline receive the same native candidates. This measures how each handles the sound after resolving the query. Test Roman spelling generalization separately with the pilot aliases switched off and a held-out spelling set.


In [ ]:
def roman_key(value):
    return " ".join(re.findall(r"[a-z0-9]+", value.casefold()))

# Optional local file from the official Dakshina release, with three tab-separated
# columns per row. Keep the upstream train/dev/test partitions intact.
DAKSHINA_TRAIN = WORK / "ur.translit.sampled.train.tsv"
ROMAN_WORDS = defaultdict(Counter)
if DAKSHINA_TRAIN.is_file():
    for line in DAKSHINA_TRAIN.read_text(encoding="utf-8").splitlines():
        if line.strip():
            native, roman, votes = line.split("\t")
            ROMAN_WORDS[roman_key(roman)][normalize(native)] += int(votes)

PILOT_ALIASES = {roman_key(alias): native for native, a, b in QUERIES for alias in (a, b)}

def resolve_query(query, use_pilot_aliases=True, limit=3):
    if re.search(r"[\u0600-\u06ff]", query):
        # Mixed-script phrases need an explicit alias or a bilingual resolver in v1.
        return [normalize(query)]
    key = roman_key(query)
    if use_pilot_aliases and key in PILOT_ALIASES:
        return [normalize(PILOT_ALIASES[key])]
    if not key:
        return []
    choices = [(0.0, ())]
    for word in key.split():
        counts = ROMAN_WORDS.get(word)
        if not counts:
            return []
        total = sum(counts.values())
        choices = sorted(
            ((score + math.log(count / total), phrase + (native,))
             for score, phrase in choices for native, count in counts.most_common(limit)),
            key=lambda x: -x[0],
        )[:limit]
    return [" ".join(parts) for _, parts in choices]

from difflib import SequenceMatcher

def fuzzy_score(query, transcript):
    words = normalize(transcript).split()
    q = normalize(query)
    if not q or not words:
        return 0.0
    size = len(q.split())
    return max((SequenceMatcher(None, q, " ".join(words[i:i+n]), autojunk=False).ratio()
                for n in range(max(1, size-1), min(len(words), size+1)+1)
                for i in range(len(words)-n+1)), default=0.0)

def read_predictions(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        entries = list(csv.DictReader(handle))
    names = [row["audio_file"] for row in entries]
    if len(set(names)) != len(names):
        raise ValueError("Each audio_file needs one prediction.")
    indexed = {row["audio_file"]: row["text"] for row in entries}
    missing = {x[1] for x in EXAMPLES} - indexed.keys()
    if missing:
        raise ValueError(f"Missing Zipformer predictions for {sorted(missing)}")
    return indexed

def transcript_scores(predictions, candidates):
    return {wav: max((fuzzy_score(q, predictions[wav]) for q in candidates), default=0.0)
            for _, wav, *_ in EXAMPLES}

def ranking_credit(scores, correct_wav):
    # Split top-rank credit equally on ties, so empty output cannot win by file order.
    if not scores or max(scores.values()) <= 0:
        return 0.0
    best = max(scores.values())
    tied = {k for k, v in scores.items() if np.isclose(v, best, rtol=0, atol=1e-8)}
    return float(correct_wav in tied) / len(tied)

prediction_path = WORK / "zipformer_predictions.csv"
if prediction_path.exists():
    predictions = read_predictions(prediction_path)
    credits = [ranking_credit(transcript_scores(predictions, resolve_query(a)), wav)
               for _, wav, _, a, _ in EXAMPLES]
    print(f"Zipformer fuzzy search, tie-adjusted Recall@1 on eight present queries: {np.mean(credits):.3f}")
else:
    print("The optional Zipformer comparison runs when zipformer_predictions.csv is present.")


## 4 · Train the acoustic student

The model uses 64 log-mel bands, a stride-two convolution, three forward GRU layers of width 256 and a character CTC head. It learns from utterance transcripts. CTC is connectionist temporal classification, a loss that can align a character sequence to audio without word timestamps. Each output frame is roughly 20 ms apart. The per-clip feature normalization in this notebook uses the full clip, so this version indexes recordings offline.

The default run uses the public FLEURS Urdu training set. The notebook holds out training **sentence IDs** for model selection and leaves all the original FLEURS test recordings untouched. For a larger run, set `LOCAL_TSV` to your own training manifest, for example one prepared from the Kathbath Urdu training split. Use a UTF-8, tab-delimited file with headers `audio_path`, `text`, `speaker_id`. Audio paths point to WAV files and can be relative to the TSV. The local-data path sets aside speakers and removes duplicate normalized prompts shared with that validation set. Keep your final test speakers outside this file. Log-mel features stay on disk for the larger run. FLEURS read speech is a starting point for the voice-note experiment.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

RATE, HOP, WIN, FFT, BANDS = 16_000, 160, 400, 512, 64
mel_edges = np.linspace(0, 2595 * np.log10(1 + (RATE / 2) / 700), BANDS + 2)
hz = 700 * (np.power(10, mel_edges / 2595) - 1)
freq = np.fft.rfftfreq(FFT, 1 / RATE)[None, :]
left = (freq - hz[:-2, None]) / (hz[1:-1, None] - hz[:-2, None])
right = (hz[2:, None] - freq) / (hz[2:, None] - hz[1:-1, None])
filters = np.maximum(0, np.minimum(left, right)).astype(np.float32)
window = np.hanning(WIN).astype(np.float32)

def read_audio(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", wavfile.WavFileWarning)
        sr, raw = wavfile.read(path)
    if np.issubdtype(raw.dtype, np.unsignedinteger):
        scale = (np.iinfo(raw.dtype).max + 1) / 2
        signal = (raw.astype(np.float32) - scale) / scale
    elif np.issubdtype(raw.dtype, np.signedinteger):
        signal = raw.astype(np.float32) / float(-np.iinfo(raw.dtype).min)
    else:
        signal = raw.astype(np.float32)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    if sr != RATE:
        g = math.gcd(sr, RATE)
        signal = resample_poly(signal, RATE // g, sr // g)
    if signal.size == 0:
        raise ValueError(f"Empty audio: {path}")
    return signal.astype(np.float32)

def features(path):
    y = read_audio(path)
    if len(y) < WIN:
        y = np.pad(y, (0, WIN - len(y)))
    frames = np.lib.stride_tricks.sliding_window_view(y, WIN)[::HOP] * window
    power = np.abs(np.fft.rfft(frames, FFT)).astype(np.float32) ** 2
    feat = np.log(np.maximum(power @ filters.T, 1e-6))
    return ((feat - feat.mean(0)) / (feat.std(0) + 1e-4)).astype(np.float32)

class AcousticStudent(nn.Module):
    def __init__(self, n_symbols):
        super().__init__()
        self.conv = nn.Conv1d(BANDS, 128, kernel_size=5, stride=2, padding=2)
        self.rnn = nn.GRU(128, 256, num_layers=3, batch_first=True, dropout=0.1)
        self.head = nn.Linear(256, n_symbols)

    def forward(self, features_batch):
        encoded = F.gelu(self.conv(features_batch.transpose(1, 2))).transpose(1, 2)
        hidden, _ = self.rnn(encoded)
        return self.head(hidden).log_softmax(-1)

def transcript_symbols(text):
    return "|" + normalize(text).replace(" ", "|") + "|"

def encode(text, ids):
    return [ids[c] for c in transcript_symbols(text)]

def pack_batch(samples):
    feats, targets = zip(*samples)
    xlen = torch.tensor([(len(f) + 1) // 2 for f in feats], dtype=torch.long)
    ylen = torch.tensor([len(t) for t in targets], dtype=torch.long)
    return pad_sequence(feats, batch_first=True), torch.cat(targets), xlen, ylen

print(f"Training device: {device}.")


In [ ]:
EPOCHS = 10
BATCH = 12
LOCAL_TSV = None  # Example: Path("kathbath_urdu_train.tsv")
run_tag = "fleurs"
if LOCAL_TSV is not None:
    LOCAL_TSV = Path(LOCAL_TSV).expanduser().resolve(strict=True)
    run_tag = "local_" + digest(LOCAL_TSV)[:12]
checkpoint = WORK / f"findword_student_{run_tag}.pt"
feature_cache = WORK / f"features_{run_tag}"
feature_cache.mkdir(exist_ok=True)

def training_source():
    rng = random.Random(SEED)
    if LOCAL_TSV is None:
        table = read_manifest("train")
        sentences = sorted({m["sentence_id"] for m in table.values()})
        rng.shuffle(sentences)
        holdout = set(sentences[:max(1, len(sentences) // 12)])
        val_names = [n for n, r in table.items() if r["sentence_id"] in holdout]
        fit_names = [n for n, r in table.items() if r["sentence_id"] not in holdout]
        directory = fetch_wavs("train", table)
        return table, fit_names, val_names, directory
    table = {}
    with LOCAL_TSV.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle, delimiter="\t"):
            wav = str((LOCAL_TSV.parent / row["audio_path"]).resolve(strict=True))
            speaker = row["speaker_id"].strip()
            if not speaker or wav in table:
                raise ValueError("Each WAV needs one row and a nonempty speaker_id.")
            table[wav] = {"text": row["text"], "speaker_id": speaker}
    speakers = sorted({r["speaker_id"] for r in table.values()})
    if len(speakers) < 2:
        raise ValueError("The local training manifest requires at least two speakers.")
    rng.shuffle(speakers)
    holdout = set(speakers[:max(1, len(speakers) // 10)])
    val_names = [n for n, r in table.items() if r["speaker_id"] in holdout]
    heldout_prompts = {normalize(table[n]["text"]) for n in val_names}
    fit_names = [n for n, r in table.items() if r["speaker_id"] not in holdout
                 and normalize(r["text"]) not in heldout_prompts]
    if not fit_names:
        raise ValueError("No training prompts remain after speaker and repeated-prompt separation.")
    print(f"Local split: {len(fit_names)} training recordings, {len(val_names)} validation recordings.")
    return table, fit_names, val_names, Path(".")

class FeatureRows(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        path, targets = self.rows[idx]
        return torch.from_numpy(np.load(path, allow_pickle=False)), targets

def make_examples(table, names, directory, ids):
    rows = []
    dropped = Counter()
    for name in names:
        text = table[name]["text"]
        if not normalize(text):
            dropped["empty text"] += 1
            continue
        if not set(transcript_symbols(text)) <= ids.keys():
            dropped["out-of-vocabulary character"] += 1
            continue
        target = encode(text, ids)
        source = directory / name
        stat = source.stat()
        fingerprint = f"{source.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|mel64_v1"
        key = hashlib.sha256(fingerprint.encode()).hexdigest()
        cached = feature_cache / (key + ".npy")
        if not cached.exists():
            with tempfile.NamedTemporaryFile(dir=feature_cache, delete=False) as out:
                tmp = Path(out.name)
                try:
                    np.save(out, features(source), allow_pickle=False)
                except BaseException:
                    tmp.unlink(missing_ok=True)
                    raise
            os.replace(tmp, cached)
        frame_count = np.load(cached, mmap_mode="r", allow_pickle=False).shape[0]
        min_frames = len(target) + sum(a == b for a, b in zip(target, target[1:]))
        if (frame_count + 1) // 2 < min_frames:
            dropped["insufficient CTC frames"] += 1
            continue
        rows.append((cached, torch.tensor(target, dtype=torch.long)))
    if not rows:
        raise ValueError("No usable audio examples.")
    print(f"Kept {len(rows)} clips; filtered {dict(dropped)}.")
    return FeatureRows(rows)

def run_epoch(model, loader, opt=None):
    model.train(opt is not None)
    loss_total, n = 0.0, 0
    with torch.set_grad_enabled(opt is not None):
        for x, targets, xl, yl in loader:
            logits = model(x.to(device)).transpose(0, 1).contiguous()
            loss = F.ctc_loss(logits, targets.to(device), xl, yl, blank=0, reduction="mean")
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite CTC loss.")
            if opt is not None:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
            loss_total += loss.item() * len(x)
            n += len(x)
    return loss_total / n

if not checkpoint.exists():
    train_meta, fit_names, val_names, train_dir = training_source()
    alphabet = sorted({c for n in fit_names for c in transcript_symbols(train_meta[n]["text"])})
    ids = {c: i+1 for i, c in enumerate(alphabet)}
    fit = make_examples(train_meta, fit_names, train_dir, ids)
    val = make_examples(train_meta, val_names, train_dir, ids)
    fit_loader = DataLoader(fit, batch_size=BATCH, shuffle=True, collate_fn=pack_batch)
    val_loader = DataLoader(val, batch_size=BATCH, collate_fn=pack_batch)
    student = AcousticStudent(len(ids)+1).to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=7e-4)
    best = math.inf
    for epoch in range(1, EPOCHS+1):
        train_loss = run_epoch(student, fit_loader, opt)
        val_loss = run_epoch(student, val_loader)
        print(f"Epoch {epoch:02d}  train {train_loss:.3f}  held-out loss {val_loss:.3f}")
        if val_loss < best:
            best = val_loss
            partial = checkpoint.with_suffix(".tmp")
            torch.save({"weights": student.state_dict(), "symbols": ids, "epoch": epoch}, partial)
            os.replace(partial, checkpoint)

state = torch.load(checkpoint, map_location="cpu", weights_only=True)
ids = state["symbols"]
student = AcousticStudent(len(ids)+1)
student.load_state_dict(state["weights"])
student.eval()
torch.set_num_threads(1)
n_parameters = sum(p.numel() for p in student.parameters())
print(f"Loaded epoch {state['epoch']}. {n_parameters:,} trainable parameters. CPU inference uses one thread.")


## 5 · Find a phrase directly in the acoustic scores

A new CTC path can start at any frame. At each frame, dynamic programming considers staying on the same symbol, moving to the next symbol, or skipping an optional blank between different symbols. Leading and trailing word separators discourage substring hits. The score compares the query path with the locally best symbol and charges a small cost for duration. It is a ranking score. Section 6 calibrates a rejection threshold independently.

The start and end include boundary symbols. They are model proposals for listening and annotation.


In [ ]:
FRAME_SEC = 2 * HOP / RATE
FRAME_WINDOW_SEC = WIN / RATE

def ctc_span(logp, tokens):
    """Return (similarity, first_frame, last_frame) for the best contiguous CTC path."""
    tokens = np.asarray(tokens, dtype=np.int64)
    if len(tokens) == 0 or len(logp) == 0 or np.any(tokens == 0):
        return 0.0, None, None
    states = np.zeros(2 * len(tokens) - 1, dtype=np.int64)
    states[::2] = tokens
    skip = np.zeros(len(states), dtype=bool)
    skip[2:] = (states[2:] != 0) & (states[2:] != states[:-2])
    prev = np.full(len(states), -np.inf)
    begins = np.zeros(len(states), dtype=np.int64)
    best, begin, end = -np.inf, None, None
    # Vectorize emission lookups across the clip; the time recurrence stays ordered.
    emissions = logp[:, states] - np.max(logp, axis=1, keepdims=True) - 0.03
    for t, emission in enumerate(emissions):
        value, start = prev.copy(), begins.copy()
        move = prev[:-1] > value[1:]
        value[1:] = np.where(move, prev[:-1], value[1:])
        start[1:] = np.where(move, begins[:-1], start[1:])
        jump = skip[2:] & (prev[:-2] > value[2:])
        value[2:] = np.where(jump, prev[:-2], value[2:])
        start[2:] = np.where(jump, begins[:-2], start[2:])
        if value[0] <= 0.0:
            value[0], start[0] = 0.0, t
        prev, begins = value + emission, start
        if prev[-1] > best:
            best, begin, end = float(prev[-1]), int(begins[-1]), t
    if not np.isfinite(best):
        return 0.0, None, None
    return math.exp(min(0.0, best / len(tokens))), begin, end

def index_audio(path):
    feat = torch.from_numpy(features(path))[None, :, :]
    with torch.inference_mode():
        return student(feat)[0].cpu().numpy()

def greedy_text(logp):
    symbols = {v: k for k, v in ids.items()}
    sequence = (x for x, _ in itertools.groupby(logp.argmax(1).tolist()) if x)
    return "".join(symbols[x] for x in sequence).replace("|", " ").strip()

def acoustic_hit(logp, candidates):
    best = (0.0, None, None, None)
    for native in candidates:
        if not normalize(native) or not set(transcript_symbols(native)) <= ids.keys():
            continue
        score, first, last = ctc_span(logp, encode(native, ids))
        if first is not None and (best[1] is None or score > best[0]):
            best = (score, first*FRAME_SEC, last*FRAME_SEC + FRAME_WINDOW_SEC, native)
    return best

def build_index(directory, filenames):
    return {name: index_audio(directory / name) for name in filenames}

index = build_index(dev_dir, [x[1] for x in EXAMPLES])
student_transcripts = {name: greedy_text(logp) for name, logp in index.items()}

def rank_audio(query, audio_index, use_pilot_aliases=True):
    candidates = resolve_query(query, use_pilot_aliases=use_pilot_aliases)
    if not candidates:
        return []
    results = []
    for name, logp in audio_index.items():
        score, start, end, candidate = acoustic_hit(logp, candidates)
        if start is not None:
            results.append({"audio_file": name, "native_candidate": candidate, "score": score,
                            "start_s": start, "end_s": end})
    return sorted(results, key=lambda row: -row["score"])

typed = "moong phali"
results = rank_audio(typed, index)
print(f"Query {typed!r}; expected source file for this pilot: {demo_wav}")
for row in results[:3]:
    print(f"{row['score']:.3f}   {row['start_s']:.2f}–{row['end_s']:.2f} s   {row['audio_file']}")
if results:
    hit = results[0]
    y = read_audio(dev_dir / hit["audio_file"])
    first = max(0, int((hit["start_s"] - .25)*RATE))
    last = min(len(y), int((hit["end_s"] + .25)*RATE))
    print("Listen to the top proposed span with 0.25 seconds of context on each side.")
    if last > first:
        display(Audio(y[first:last], rate=RATE))


## 6 · Measure retrieval and false returns

The calibration index uses eight **different** development sentence IDs. Its queries come from more development sentences and are absent from every calibration transcript. Calibration uses native candidates, as the pilot aliases each resolve to a single native spelling. The maximum over all eight clips is the score for one no-match query. Pick separate thresholds for the acoustic score and the same student's greedy transcript plus fuzzy search. Both get the same calibration examples and false-return budget.

The pilot outputs predicted intervals. Timing accuracy requires someone to listen and mark the boundaries. Four absent pilot queries give only a coarse check of rejection.


In [ ]:
def choose_calibration(table, count=40):
    excluded_ids = {e[0] for e in EXAMPLES}
    by_id = {}
    for wav, row in sorted(table.items()):
        if row["sentence_id"] not in excluded_ids:
            by_id.setdefault(row["sentence_id"], wav)
    names = list(by_id.values())
    rng = random.Random(SEED)
    rng.shuffle(names)
    controls = names[:len(EXAMPLES)]
    controls_text = [table[name]["text"] for name in controls]
    control_words = set(" ".join(normalize(t) for t in controls_text).split())
    excluded_queries = {normalize(x[0]) for x in QUERIES}
    candidate_groups = defaultdict(list)
    for wav in names[len(controls):]:
        words = normalize(table[wav]["text"]).split()
        for length in (1, 2, 3):
            found = set()
            for i in range(len(words)-length+1):
                chunk = words[i:i+length]
                q = " ".join(chunk)
                if (q in excluded_queries or not re.search(r"[\u0600-\u06ff]", q)
                        or not set(transcript_symbols(q)) <= ids.keys()
                        or any(phrase_in(q, ref) for ref in controls_text)):
                    continue
                found.add((q, bool(set(chunk) & control_words)))
            for q, overlaps in sorted(found):
                candidate_groups[(length, overlaps)].append(q)
    chosen = []
    lengths = [len(normalize(native).split()) for native, _, _ in QUERIES]
    for i in range(count):
        length = lengths[i % len(lengths)]
        prefer_overlap = length > 1 and i % 2 == 0
        pool = candidate_groups[length, prefer_overlap] + candidate_groups[length, not prefer_overlap]
        if not pool:
            continue
        # Prefer lexical near-misses for half the multi-word queries where available.
        first_pool = [q for q in candidate_groups[length, prefer_overlap] if q not in chosen]
        choices = first_pool or [q for q in pool if q not in chosen]
        if choices:
            chosen.append(rng.choice(choices))
    if len(controls) != len(EXAMPLES) or len(chosen) != count:
        raise ValueError("Insufficient disjoint calibration material.")
    return controls, chosen

def acceptance_cutoff(scores, budget=0.05):
    ordered = sorted(scores, reverse=True)
    if not ordered or not 0 <= budget < 1:
        raise ValueError("Supply negative calibration scores and a budget in [0, 1).")
    return float(np.nextafter(ordered[math.floor(budget*len(ordered))], math.inf))

control_names, no_match_queries = choose_calibration(dev)
control_dir = fetch_wavs("dev", control_names)
control_index = build_index(control_dir, control_names)
control_hyps = {n: greedy_text(s) for n, s in control_index.items()}

def negative_maxima(audio_index, hypotheses, native_queries):
    acoustic, fuzzy = [], []
    for q in native_queries:
        acoustic.append(max(acoustic_hit(x, [q])[0] for x in audio_index.values()))
        fuzzy.append(max(fuzzy_score(q, t) for t in hypotheses.values()))
    return acoustic, fuzzy

ac_scores, text_scores = negative_maxima(control_index, control_hyps, no_match_queries)
ac_cutoff, text_cutoff = acceptance_cutoff(ac_scores), acceptance_cutoff(text_scores)
print(f"Calibrated on {len(ac_scores)} absent queries against a different eight-clip index.")
print(f"Cutoffs at 5% empirical false-return budget: acoustic {ac_cutoff:.4f}, fuzzy {text_cutoff:.4f}")


In [ ]:
expected_file = {native: wav for _, wav, native, *_ in EXAMPLES}
metrics = {name: {"rank": [], "accepted_correct": [], "false_return": []}
           for name in ("acoustic", "greedy + fuzzy")}
out = []
for native, roman, _ in QUERIES:
    candidates = resolve_query(roman)
    hits = rank_audio(roman, index)
    acoustic_scores = {row["audio_file"]: row["score"] for row in hits}
    fuzzy_scores = transcript_scores(student_transcripts, candidates)
    for method, scores, threshold in (("acoustic", acoustic_scores, ac_cutoff),
                                      ("greedy + fuzzy", fuzzy_scores, text_cutoff)):
        max_score = max(scores.values(), default=0)
        accepted = bool(scores) and max_score >= threshold
        correct = expected_file.get(native)
        if correct:
            credit = ranking_credit(scores, correct)
            metrics[method]["rank"].append(credit)
            metrics[method]["accepted_correct"].append(credit * accepted)
        else:
            metrics[method]["false_return"].append(accepted)
    for row in hits:
        out.append({"query": roman, **row, "accepted": row["score"] >= ac_cutoff,
                    "present_in_reference": int(row["audio_file"] == expected_file.get(native))})
for method, stat in metrics.items():
    print(f"{method:16}  R@1 {np.mean(stat['rank']):.2f}  "
          f"accepted correct / 8 present {np.mean(stat['accepted_correct']):.2f}  "
          f"false returns {sum(stat['false_return'])}/{len(stat['false_return'])}")
if out:
    write_rows(WORK / "pilot_predictions.csv", list(out[0]), out)

# Optional personal demo. Set a local WAV path and a query, then rerun this cell.
CUSTOM_WAV = None
CUSTOM_QUERY = "مونگ پھلی"
if CUSTOM_WAV:
    custom = Path(CUSTOM_WAV)
    print(rank_audio(CUSTOM_QUERY, {custom.name: index_audio(custom)})[:1])
    print("The eight-recording pilot threshold applies to that workload. Calibrate on your own index before using rejection.")




### Sources

- Conneau et al., FLEURS. The seed refers to the Pakistani Urdu development split. The source is licensed CC BY 4.0. Changes here consist of query selections, pilot Roman spellings and phrase-presence labels. [Dataset card](https://huggingface.co/datasets/google/fleurs) · [Urdu development metadata](https://huggingface.co/datasets/google/fleurs/blob/main/data/ur_pk/dev.tsv) · [Development audio](https://huggingface.co/datasets/google/fleurs/blob/main/data/ur_pk/audio/dev.tar.gz)
- Javed et al., IndicSUPERB and Kathbath. AI4Bharat describes CC0 packaging and identifies IndicCorp as the source of the prompt text. [Benchmark](https://github.com/AI4Bharat/IndicSUPERB) · [Dataset and access conditions](https://huggingface.co/datasets/ai4bharat/Kathbath)
- Roark et al., Dakshina. The text data are licensed CC BY-SA 4.0. No Dakshina rows are bundled with this notebook. [Paper](https://aclanthology.org/2020.lrec-1.294/) · [Data and download](https://github.com/google-research-datasets/dakshina)
- Urdu spoken-term detection precedes this project, and CTC alignment for user-supplied text has also been published. [Urdu spoken-term detection](https://www.cle.org.pk/clt20/papers/Paper-4.pdf) · [Jin et al., Interspeech 2024](https://www.isca-archive.org/interspeech_2024/jin24d_interspeech.html) · [Multilingual spoken-query search using transliteration](https://www.isca-archive.org/interspeech_2025/r25_interspeech.html)
- The existing Urdu streaming Zipformer supplies a practical baseline. The smol-audio project demonstrates the reproducible notebook release format. [Urdu Streaming ASR](https://github.com/mwzkhalil/Urdu-Streaming-ASR) · [smol-audio](https://github.com/Deep-unlearning/smol-audio)
